# 05c JEV v4 Score edge trials

Frozen validation spec v1.2 SHA256: `1e0920672741dc86f97f0dc241c8d286b9ab6067ec976c993b8a9b7a6865600e`. Validator SHA256: `c3b905d046fe568e5e0a35f59d30b07361eb9f3961d39594bba12b2a3386bec9`.

Completed 10 sequential, paired trials with the fixed 20 features, train-only Robust 1x inputs, and exact v4 question in each same-window request. All ten candidates were rejected. The same development windows were reused, so results are exploratory and selection-biased.

The v1.2 numeric-boundary fix was replayed over every stored response before trial 5; thresholds, metric definitions, samples, and group folds stayed fixed. No untouched eligible groups remain in the FlowCool-leak/near-failure cell, so final confirmation and group-paired bootstrap are inconclusive. TTF bands are surrogates for current condition; balanced-sample precision and PR-AUC do not represent operating prevalence.


In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().resolve()
OUT = ROOT / 'artifacts/jev_v4_edge_trials'
summary = pd.read_csv(OUT / 'trial_summary.csv')
registry = pd.read_csv(OUT / 'hypothesis_registry.csv')
final = json.loads((OUT / 'final_confirmation.json').read_text(encoding='utf-8'))


## Trial summary

The validator returns per-target ROC-AUC and OOF macro-F1, pooled and per-target dangerous errors, fold-specific training thresholds, every provisional-rule condition, and leave-one-group-out changes. Full records are saved in `artifacts/jev_v4_edge_trials/validations/`.


In [ ]:
summary


## Per-target results

This table keeps each target's candidate metric and dangerous-error counts visible. Compare it with the paired v4 values in each validation JSON.


In [ ]:
details = []
for path in sorted((OUT / 'validations').glob('trial_*.json')):
    result = json.loads(path.read_text(encoding='utf-8'))
    if not result.get('metrics'): continue
    for target, metric in result['metrics']['candidate']['per_target'].items():
        details.append({'trial': result['trial'], 'mechanism_id': result['mechanism_id'], 'target': target,
            'roc_auc': metric['roc_auc_normal_vs_near_failure'], 'oof_macro_f1': metric['oof_macro_f1'],
            'normal_to_high': metric['dangerous_errors']['normal_to_high']['count'],
            'near_failure_to_low': metric['dangerous_errors']['near_failure_to_low']['count']})
per_target_results = pd.DataFrame(details)
per_target_results


## Final confirmation

No candidate passed the development-only provisional keep rule. There were zero unused groups for the required leak/near-failure cell, so the locked unseen-group evaluation and paired group bootstrap could not be run. This result is not `VERIFIED`.


In [ ]:
final
